## Module C — Visualization & insights (**daily**, corrected)\n
\n
This notebook updates the visualization module to use **daily aligned data** (not annual means):\n
\n
- Dual-axis **normalized price** trends (GLD vs NFLX)\n
- Scatter of **daily returns** with regression fit line\n
- Correlation heatmap (returns)\n
- Rolling correlation (to show dependence changes over time)\n
- Tail-day highlighting (worst 5% NFLX days)\n
\n
Input files (from Module A daily or the script output):\n
- `outputs/combined_daily_prices.csv`\n
- `outputs/combined_daily_logreturns.csv`\n

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

ROOT = Path.cwd()
# Some tools set the notebook working directory to `notebooks/`.
# Also, an empty `notebooks/outputs/` folder might exist; therefore we check for the actual data files.
if not (ROOT / "outputs" / "combined_daily_prices.csv").exists():
    ROOT = ROOT.parent

OUT_DIR = ROOT / "outputs"
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

prices_path = OUT_DIR / "combined_daily_prices.csv"
rets_path = OUT_DIR / "combined_daily_logreturns.csv"

prices = pd.read_csv(prices_path)
rets = pd.read_csv(rets_path)
prices["Date"] = pd.to_datetime(prices["Date"])
rets["Date"] = pd.to_datetime(rets["Date"])

prices.head(), rets.head()


FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\ASUS\\Desktop\\foundation of analysis\\final_project_0\\notebooks\\outputs\\combined_daily_prices.csv'

### Dual-axis normalized price trends\n
\n
Because NFLX and GLD are on different price scales, we normalize each series to 1.0 at the first date.\n

In [ ]:
sns.set_style("whitegrid")

p = prices.sort_values("Date").copy()

p["NFLX_norm"] = p["NFLX_AdjClose"] / p["NFLX_AdjClose"].iloc[0]

p["GLD_norm"] = p["GLD_NAV"] / p["GLD_NAV"].iloc[0]



fig, ax1 = plt.subplots(figsize=(12, 5))

ax1.plot(p["Date"], p["GLD_norm"], color="tab:gold", linewidth=1.8, label="GLD (normalized)")

ax1.set_ylabel("GLD normalized", color="tab:gold")

ax1.tick_params(axis="y", labelcolor="tab:gold")



ax2 = ax1.twinx()

ax2.plot(p["Date"], p["NFLX_norm"], color="tab:blue", linewidth=1.8, label="NFLX (normalized)")

ax2.set_ylabel("NFLX normalized", color="tab:blue")

ax2.tick_params(axis="y", labelcolor="tab:blue")



fig.suptitle("Daily normalized trend: GLD vs NFLX")

fig.tight_layout()

out = FIG_DIR / "dual_axis_trend_daily_normalized.png"

fig.savefig(out, dpi=200, bbox_inches="tight")

out


### Scatter of daily returns + regression fit line\n

In [ ]:
x = rets["GLD_logret"].to_numpy()

y = rets["NFLX_logret"].to_numpy()

X = sm.add_constant(x)

res = sm.OLS(y, X, missing="drop").fit(cov_type="HC1")



grid = np.linspace(np.nanmin(x), np.nanmax(x), 200)

yhat = res.params[0] + res.params[1] * grid



fig, ax = plt.subplots(figsize=(7.5, 5.5))

ax.scatter(x, y, s=10, alpha=0.25)

ax.plot(grid, yhat, color="crimson", linewidth=2, label=f"OLS (HC1) slope={res.params[1]:.4f}")

ax.set_xlabel("GLD daily log return")

ax.set_ylabel("NFLX daily log return")

ax.set_title("Daily returns scatter (with OLS fit)")

ax.legend()

fig.tight_layout()

out = FIG_DIR / "scatter_returns_with_fit_daily.png"

fig.savefig(out, dpi=200, bbox_inches="tight")

out


### Correlation heatmap (returns)\n

In [ ]:
corr = rets[["NFLX_logret", "GLD_logret"]].corr()

fig, ax = plt.subplots(figsize=(5, 4))

sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1, ax=ax)

ax.set_title("Correlation heatmap (daily log returns)")

fig.tight_layout()

out = FIG_DIR / "correlation_heatmap_returns_daily.png"

fig.savefig(out, dpi=200, bbox_inches="tight")

out


### Rolling correlation (dependence is time-varying)\n

In [ ]:
window = 126  # ~6 months trading days

tmp = rets.sort_values("Date").copy()

tmp["rolling_corr"] = tmp["NFLX_logret"].rolling(window).corr(tmp["GLD_logret"])



fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(tmp["Date"], tmp["rolling_corr"], linewidth=1.5)

ax.axhline(0, color="black", linewidth=1, alpha=0.6)

ax.set_title(f"Rolling correlation (window={window} days)")

ax.set_ylabel("corr(NFLX, GLD)")

fig.tight_layout()

out = FIG_DIR / "rolling_corr_daily.png"

fig.savefig(out, dpi=200, bbox_inches="tight")

out


### Tail-day highlighting (worst 5% NFLX days)\n

In [ ]:
q = 0.05

cutoff = rets["NFLX_logret"].quantile(q)

tail = rets[rets["NFLX_logret"] <= cutoff].copy()



fig, ax = plt.subplots(figsize=(7.5, 5.5))

ax.scatter(rets["GLD_logret"], rets["NFLX_logret"], s=10, alpha=0.15, label="All days")

ax.scatter(tail["GLD_logret"], tail["NFLX_logret"], s=18, alpha=0.6, label=f"Tail days (NFLX bottom {int(q*100)}%)")

ax.set_xlabel("GLD daily log return")

ax.set_ylabel("NFLX daily log return")

ax.set_title("Tail highlighting: dependence may differ in crashes")

ax.legend()

fig.tight_layout()

out = FIG_DIR / "tail_highlight_scatter_daily.png"

fig.savefig(out, dpi=200, bbox_inches="tight")

out
